### install necessary library

In [ ]:
import pandas as pd
import requests
import time
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
ETHERSCAN_API_KEY = os.environ.get("ETHERSCAN_API_KEY", "")

RAW_DATA_DIR = os.path.join("raw_data") + os.sep
os.makedirs(RAW_DATA_DIR, exist_ok=True)

MAX_SAMPLES = 200

CHAIN_MAP = {
    'eth': '1',
    'bsc': '56'
}


### Download Ground Truths lists:

rugpull\_dataset\.csv for the "Bad" guys

verified\_safe\.json for the "Good" guys

In [21]:
def download_ground_truth():
    print("Checking for existing ground truth files...")
    
    rug_path = os.path.join(RAW_DATA_DIR, "rugpull_dataset.csv")
    safe_path = os.path.join(RAW_DATA_DIR, "verified_safe.json")

    # 1.Check/Download Rugpull List
    if os.path.exists(rug_path):
        print(f"Found existing '{rug_path}'. Skipping download.")
    else:
        print("Downloading rugpull list...")
        rug_url = "https://raw.githubusercontent.com/dianxiang-sun/rug_pull_dataset/main/rugpull_dataset.csv"
        rug_df = pd.read_csv(rug_url)
        rug_df.to_csv(rug_path, index=False)
    
    # 2.Check/Download Safe List
    if os.path.exists(safe_path):
        print(f"Found existing '{safe_path}'. Skipping download.")
    else:
        print("Downloading verified safe list...")
        safe_url = "https://gateway.ipfs.io/ipns/tokens.uniswap.org"
        safe_data = requests.get(safe_url).json()
        with open(safe_path, 'w') as f:
            json.dump(safe_data, f)

### Transaction Fetching Agent \(RPC API\)

In [23]:
def fetch_blockchain_data(address, chain, label):
    """MCP-style Agent: Pulls raw transaction hashes from the ledger."""
    base_url = "https://api.etherscan.io/v2/api"
    chain_id = "1" if chain == 'eth' else "56"
    
    # Normalize chain string
    chain = str(chain).strip().lower()

    # Convert to official chain ID
    chain_id = CHAIN_MAP.get(chain)

    # Skip unsupported chains
    if not chain_id:
        print(f"Unsupported chain: {chain}")
        return False
        

    params = {
        "chainid": chain_id,
        "module": "account",
        "action": "tokentx",
        "contractaddress": address,
        "page": 1,
        "offset": 1000, # Forensic window
        "sort": "asc",
        "apikey": ETHERSCAN_API_KEY
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        if data['status'] == '1' and len(data['result']) > 0:
            df = pd.DataFrame(data['result'])
            # Label the filename so processing.ipynb knows what it is
            status = "fraud" if label == 1 else "safe"
            filename = f"{RAW_DATA_DIR}{status}_{chain}_{address}.csv"
            df.to_csv(filename, index=False)
            return True
        return False
    except:
        return False

In [25]:
download_ground_truth()

Checking for existing ground truth files...
Found existing '/work/01_Data_Pipeline/raw_data/rugpull_dataset.csv'. Skipping download.
Found existing '/work/01_Data_Pipeline/raw_data/verified_safe.json'. Skipping download.


### Blockchain Extraction

- Active Forensic Footprints \(Success\)

- The contract has active ERC\-20 \(Ethereum\) or BEP\-20 \(BSC\) token transfers, containing the sender/receiver data needed to build the NetworkX graph\.

- Native\-Only Wallet \(Failed\)

- The address only moves native gas tokens \(ETH or BNB\) rather than specific meme\-coin tokens\. These are filtered out as they lack the "Token Topology" required for meme\-coin analysis\.

- "Ghost" or Inactive Scams \(No Data\)

- The contract was deployed to the blockchain but never "launched" \(no liquidity added, no trading\)\. Scammers often deploy dozens of contracts and only use the one that successfully attracts "hype"\. These provide no behavioral data and are discarded from the training set\.

- Ground\-Truth Inconsistencies \(Failed\)

- The CSV dataset incorrectly labels a BSC token as an Ethereum token\.

- Rate Limit Throttling \(Error\)

- The API key is being pinged more than 5 times per second\.

In [29]:
rug_path = os.path.join(RAW_DATA_DIR, "rugpull_dataset.csv")
rug_list = pd.read_csv(rug_path, low_memory=False)

rug_list.columns = rug_list.columns.str.strip()
rug_list = rug_list.dropna(subset=['address'])

rug_list = rug_list.sample(
    min(MAX_SAMPLES, len(rug_list)),
    random_state=42
)

# 2. Load Safe List
with open(os.path.join(RAW_DATA_DIR, "verified_safe.json"), 'r') as f:
    safe_data = json.load(f)
    safe_list = safe_data['tokens'][:MAX_SAMPLES]

print("Starting Blockchain Extraction...")


# Fetch fraud tokens
print("Fetching FRAUD samples...\n")

# --- Fetch Scams ---
for _, row in rug_list.iterrows():
    status = "fraud"
    address = str(row['address']).strip()
    chain = str(row['Chain']).strip().lower()

    if chain not in CHAIN_MAP:
        print(f"Skipping unsupported chain: {chain}")
        continue
    
    filename = os.path.join(RAW_DATA_DIR, f"{status}_{chain}_{address}.csv")
    
    if os.path.exists(filename):
        print(f"Skipping: {address[:10]}... (Already exists)")
        continue
        
    success = fetch_blockchain_data(address, chain, 1)
    
    if success: 
        print(f"Done: Fraud {address[:10]}...")
    else:
        print(f"Failed/No Data: Fraud {address[:10]}...")
        
    time.sleep(0.25) # Rate limit protection

# --- Fetch Safe ---
for token in safe_list:
    address = token['address']
    symbol = token['symbol']
    filename = os.path.join(RAW_DATA_DIR, f"safe_eth_{address}.csv")
    
    if os.path.exists(filename):
        print(f"Skipping: {symbol} (Already exists)")
        continue

    # Most Uniswap tokens are on Ethereum (eth)
    success = fetch_blockchain_data(address, 'eth', 0)
    
    if success: 
        print(f"Done: Safe {symbol}...")
    else:
        print(f"Failed: Safe {symbol}...")
        
    time.sleep(0.25)

print("\nDATA INGESTION COMPLETE. Files are stored in /raw_data")

Starting Blockchain Extraction...
Fetching FRAUD samples...

Done: Fraud 0xceb2a44f...
Done: Fraud 0x5ee9d70a...
Failed/No Data: Fraud 0x3c3f4819...
Done: Fraud 0x99dab0ae...
Done: Fraud 0x21df3b62...
Failed/No Data: Fraud 0xc5a25e92...
Done: Fraud 0xc63e6bcc...
Failed/No Data: Fraud 0x4f64c4AC...
Done: Fraud 0xbe341572...
Done: Fraud 0x2a0f041b...
Done: Fraud 0x5f7593e7...
Done: Fraud 0x97c041ac...
Done: Fraud 0x4d0a31fb...
Done: Fraud 0x42949d91...
Done: Fraud 0xa42d2cf5...
Done: Fraud 0xa940d5aa...
Done: Fraud 0x4c6335c1...
Failed/No Data: Fraud 0xf5b1d75f...
Done: Fraud 0x6c188867...
Done: Fraud 0x7e519f05...
Done: Fraud 0x834460e2...
Done: Fraud 0x2ce47cd6...
Done: Fraud 0xe460f3e0...
Done: Fraud 0xe7650a78...
Done: Fraud 0xe053cf2b...
Done: Fraud 0x7a3d70f5...
Done: Fraud 0x5882a0ed...
Done: Fraud 0xa4f4df34...
Done: Fraud 0x0ae7b758...
Done: Fraud 0x90b6082a...
Done: Fraud 0x15e10b9e...
Done: Fraud 0xa133be32...
Done: Fraud 0x19dcdc26...
Done: Fraud 0xfca1d103...
Done: Fraud 0xd

### Summary

This notebook acts as our Automated Data Fetching Agent\. It serves as the bridge between the blockchain ledger and our local environment\.

- It automatically downloads "Ground Truth" lists—known rugpulls from GitHub and verified safe tokens from Uniswap—to ensure our model knows what "good" and "bad" look like\.

- Blockchain Retrieval: Using the Etherscan V2 RPC API, it targets specific contract addresses across both Ethereum and BSC\.

Output: It captures the raw "footprints" \(transaction hashes, sender/receiver addresses and values\) and stores them as individual CSV files in the raw\_data/ folder, creating a forensic evidence locker for the project\.